In this notebook, I show some examples of how to fit with PyXspec. Note that this gives equivalent results to fitting in the command line with Xspec. At the bottom of this notebook, I give an example of code to run in the command line if you want to compare to Xspec results. 

See the note in the README.txt about the choice to include $\verb|cflux|$ after initial fitting when fitting two-component models. 

# Imports

In [ ]:
from xspec import *
import numpy as np
import matplotlib.pyplot as plt
import json
import numpy as np
import pandas as pd
from astropy.table import Table
import warnings
from astropy.time import Time 
import matplotlib as mpl

import sys
import os

sys.path.append(os.path.abspath("../CODE"))
from fit_xrt_spectra import plot_resid

# Load the Spectrum

In [ ]:
AllData.clear()
AllData('../spectra_swift_xrt/Obs_00033665136wtfinal_bin.pi') 
AllData.ignore('*:10.0-**')
AllData.ignore('*:**-0.6')
AllData.notice('*:0.6-10.0')
AllData.ignore('bad') # ignore data that is bad -- using the quality column
Xset.abund = "wilm"    # set the abundance table used in the plasma emission and photoelectric absorption models; 'wilm' is an XSPEC built-in abundance table
Xset.xsect = "vern"  
Fit.statMethod = 'cstat' 

# Fit with a Powerlaw

In [ ]:
AllModels.clear()
AllModels.systematic = 0.00 # add systematic error
mod1 = Model('tbabs * cflux* (powerlaw)')
mod1.setPars({1:'0.5 -1', 2:1.0, 3:10.0, 4:'-10', 5:'1.6,,0.0,0.0,4.0,4.0', 6:'1.0 -1'})
AllModels.show() # for checking

In [ ]:
# Fitting
Fit.statMethod = 'cstat' 
Fit.nIterations = 100  # fit iterations for each attempt
for delt in [1e-2, 1e-3, 1e-4, 1e-5]:
    Fit.delta = delt
    Fit.perform()

In [ ]:
for comp_name in mod1.componentNames: 
        comp = getattr(mod1, comp_name)
        for par_name in comp.parameterNames:
            param = getattr(comp, par_name)
            print(f'{par_name} : {param.values[0]}')

In [ ]:
n_params = mod1.nParameters
Fit.error(f'maximum 3.0 1.0 1-{n_params}')

In [ ]:
print("chi2: ", Fit.testStatistic, " dof: ", Fit.dof, " redchi2: ", Fit.testStatistic/Fit.dof)

In [ ]:
par1 = mod1.cflux.lg10Flux 

# best-fit value
val1 = par1.values[0]
#print(val1)

# bounds and ± errors
lo, hi, status = par1.error
#print(lo,hi)
err_minus1 = val1 - lo
err_plus1  = hi - val1
#print(err_minus1, err_plus1)

print(f"norm1 = {10**(val1)} (-{10**(val1) - 10**(val1 - err_minus1)}, +{10**(val1 + err_plus1) - 10**val1})")
#print(f"norm1 = {10**(val1)} (-{10**(val1) - 10**(lo)}, +{10**(hi) - 10**val1})")

# Fit with a Pegged Powerlaw
We see that the result is pretty much identical to the powerlaw one. 

In [ ]:
AllModels.clear()
AllModels.systematic = 0.00 # add systematic error
mod1 = Model('tbabs * pegpwrlw')
mod1.setPars({1:'0.5 -1', 2:'1.6,,0.0,0.0,4.0,4.0',  3:1.0, 4:10.0, 5:'1.0'})
AllModels.show() # for checking

In [ ]:
# Fitting
Fit.statMethod = 'cstat' 
Fit.nIterations = 100  # fit iterations for each attempt
for delt in [1e-2, 1e-3, 1e-4, 1e-5]:
    Fit.delta = delt
    Fit.perform()

In [ ]:
n_params = mod1.nParameters
Fit.error(f'maximum 3.0 1.0 1-{n_params}')

In [ ]:
for comp_name in mod1.componentNames: 
        comp = getattr(mod1, comp_name)
        for par_name in comp.parameterNames:
            param = getattr(comp, par_name)
            print(f'{par_name} : {param.values[0]}')

In [ ]:
print("chi2: ", Fit.testStatistic, " dof: ", Fit.dof, " redchi2: ", Fit.testStatistic/Fit.dof)

In [ ]:
par1 = mod1.pegpwrlw.norm

# best-fit value
val1 = par1.values[0]

# bounds and ± errors
lo, hi, status = par1.error
err_minus1 = val1 - lo
err_plus1  = hi - val1

print(f"norm1 = {val1*1e-12} (-{err_minus1*1e-12}, +{err_plus1*1e-12})")

In [ ]:
plot_resid("", "", mod=mod1, save=False, log=True)

---

# Fit with a Diskbb

In [ ]:
AllModels.clear()
AllModels.systematic = 0.00 # add systematic error
mod1 = Model('tbabs * cflux * diskbb')
mod1.setPars({1:'0.5 -1', 2:1.0, 3:10.0, 4:'-10', 5:'0.6,,0.05,0.05,4.0,4.0', 6:'1.0 -1'})
AllModels.show() # for checking

In [ ]:
# Fitting
Fit.statMethod = 'cstat' 
Fit.nIterations = 100  # fit iterations for each attempt
for delt in [1e-2, 1e-3, 1e-4, 1e-5]:
    Fit.delta = delt
    Fit.perform()

In [ ]:
n_params = mod1.nParameters
Fit.error(f'maximum 3.0 1.0 1-{n_params}')

In [ ]:
for comp_name in mod1.componentNames: 
        comp = getattr(mod1, comp_name)
        for par_name in comp.parameterNames:
            param = getattr(comp, par_name)
            print(f'{par_name} : {param.values[0]}')

In [ ]:
print("chi2: ", Fit.testStatistic, " dof: ", Fit.dof, " redchi2: ", Fit.testStatistic/Fit.dof)

In [ ]:
par1 = mod1.cflux.lg10Flux 

# best-fit value
val1 = par1.values[0]
#print(val1)

# bounds and ± errors
lo, hi, status = par1.error
#print(lo,hi)
err_minus1 = val1 - lo
err_plus1  = hi - val1
#print(err_minus1, err_plus1)

print(f"norm1 = {10**(val1)} (-{10**(val1) - 10**(val1 - err_minus1)}, +{10**(val1 + err_plus1) - 10**val1})")
#print(f"norm1 = {10**(val1)} (-{10**(val1) - 10**(lo)}, +{10**(hi) - 10**val1})")

In [ ]:
plot_resid("", "", mod=mod1, save=False)

---

# Fit with a Powerlaw + Diskbb

In [ ]:
AllModels.clear()
AllModels.systematic = 0.00 # add systematic error
mod1 = Model('tbabs * (powerlaw + diskbb)')
mod1.setPars({1:'0.5 -1', 2:'1.6,,0.0,0.0,4.0,4.0', 3: '1.0', 4:'0.6,,0.05,0.05,4.0,4.0', 5:'1.0'})
AllModels.show() # for checking

In [ ]:
# Fitting
Fit.statMethod = 'cstat' 
Fit.nIterations = 100  # fit iterations for each attempt
for delt in [1e-2, 1e-3, 1e-4, 1e-5]:
    Fit.delta = delt
    Fit.perform()

In [ ]:
for comp_name in mod1.componentNames: 
        comp = getattr(mod1, comp_name)
        for par_name in comp.parameterNames:
            param = getattr(comp, par_name)
            print(f'{par_name} : {param.values[0]}')

In [ ]:
powerlaw_gamma = mod1.powerlaw.PhoIndex.values[0]
print(powerlaw_gamma)
powerlaw_norm = mod1.powerlaw.norm.values[0]
print(powerlaw_norm)
diskbb_Tin = mod1.diskbb.Tin.values[0]
print(diskbb_Tin)
diskbb_norm = mod1.diskbb.norm.values[0]
print(diskbb_norm)

AllModels.clear()
AllModels.systematic = 0.00 # add systematic error
mod2 = Model('tbabs * cflux* (powerlaw + diskbb)')
mod2.setPars({1:'0.5 -1', 2:1.0, 3:10.0, 4:'-10', 5:f'{powerlaw_gamma},,0.5,0.5,4.0,4.0', 6 : f'{powerlaw_norm} -1', 7:f'{diskbb_Tin},,0.05,0.05,4.0,4.0',8:f'{diskbb_norm}' })
AllModels.show() # for checking

In [ ]:
Fit.statMethod = 'cstat' 
Fit.nIterations = 100  # fit iterations for each attempt
Fit.delta = 1e-5 # controls the convergence criterion for the fitting algorithm
Fit.perform()

In [ ]:
for comp_name in mod2.componentNames: 
        comp = getattr(mod2, comp_name)
        for par_name in comp.parameterNames:
            param = getattr(comp, par_name)
            print(f'{par_name} : {param.values[0]}')

In [ ]:
n_params = mod1.nParameters
Fit.error(f'maximum 3.0 nonew 1.0 1-{n_params}')

In [ ]:
print("chi2: ", Fit.testStatistic, " dof: ", Fit.dof, " redchi2: ", Fit.testStatistic/Fit.dof)

In [ ]:
par1 = mod2.cflux.lg10Flux 

# best-fit value
val1 = par1.values[0]
#print(val1)

# bounds and ± errors
lo, hi, status = par1.error
#print(lo,hi)
err_minus1 = val1 - lo
err_plus1  = hi - val1
#print(err_minus1, err_plus1)

print(f"norm1 = {10**(val1)} (-{10**(val1) - 10**(val1 - err_minus1)}, +{10**(val1 + err_plus1) - 10**val1})")
#print(f"norm1 = {10**(val1)} (-{10**(val1) - 10**(lo)}, +{10**(hi) - 10**val1})")

In [ ]:
plot_resid("", "", mod=mod1, save=False)

---

# Fit with a Powerlaw + BbodyRad

In [ ]:
AllModels.clear()
AllModels.systematic = 0.00 # add systematic error
mod1 = Model('tbabs * (powerlaw + bbodyrad)')
mod1.setPars({1:'0.5 -1', 2:'1.7,,0.5,0.5,4.0,4.0',3 : '1.0', 4:'0.6,,0.05,0.05,4.0,4.0',5:'1.0' })
AllModels.show() # for checking

In [ ]:
# Fitting
Fit.statMethod = 'cstat' 
Fit.nIterations = 100  # fit iterations for each attempt
for delt in [1e-2, 1e-3, 1e-4, 1e-5]:
    Fit.delta = delt
    Fit.perform()

In [ ]:
for comp_name in mod1.componentNames: 
        comp = getattr(mod1, comp_name)
        for par_name in comp.parameterNames:
            param = getattr(comp, par_name)
            print(f'{par_name} : {param.values[0]}')

In [ ]:
powerlaw_gamma = mod1.powerlaw.PhoIndex.values[0]
print(powerlaw_gamma)
powerlaw_norm = mod1.powerlaw.norm.values[0]
print(powerlaw_norm)
bbody_kT = mod1.bbodyrad.kT.values[0]
print(bbody_kT)
bbody_norm = mod1.bbodyrad.norm.values[0]
print(bbody_norm)

AllModels.clear()
AllModels.systematic = 0.00 # add systematic error
mod1 = Model('tbabs * cflux* (powerlaw + bbodyrad)')
mod1.setPars({1:'0.5 -1', 2:1.0, 3:10.0, 4:'-10', 5:f'{powerlaw_gamma},,0.5,0.5,4.0,4.0', 6 : f'{powerlaw_norm} -1', 7:f'{bbody_kT},,0.05,0.05,4.0,4.0',8:f'{bbody_norm}' })
AllModels.show() # for checking

In [ ]:
Fit.statMethod = 'cstat' 
Fit.nIterations = 100  # fit iterations for each attempt
Fit.delta = 1e-5 # controls the convergence criterion for the fitting algorithm
Fit.perform()

In [ ]:
for comp_name in mod1.componentNames: 
        comp = getattr(mod1, comp_name)
        for par_name in comp.parameterNames:
            param = getattr(comp, par_name)
            print(f'{par_name} : {param.values[0]}')

In [ ]:
n_params = mod1.nParameters
Fit.error(f'maximum 3.0 1.0 1-{n_params}')

In [ ]:
print("chi2: ", Fit.testStatistic, " dof: ", Fit.dof, " redchi2: ", Fit.testStatistic/Fit.dof)

In [ ]:
par1 = mod1.cflux.lg10Flux 

# best-fit value
val1 = par1.values[0]
#print(val1)

# bounds and ± errors
lo, hi, status = par1.error
#print(lo,hi)
err_minus1 = val1 - lo
err_plus1  = hi - val1
#print(err_minus1, err_plus1)

print(f"norm1 = {10**(val1)} (-{10**(val1) - 10**(val1 - err_minus1)}, +{10**(val1 + err_plus1) - 10**val1})")
#print(f"norm1 = {10**(val1)} (-{10**(val1) - 10**(lo)}, +{10**(hi) - 10**val1})")

In [ ]:
plot_resid("", "", mod=mod1, save=False)

---

# Fit with a Diskbb + Bbodyrad

In [ ]:
AllModels.clear()
AllModels.systematic = 0.00 # add systematic error
mod1 = Model('tbabs * (diskbb + bbodyrad)')
mod1.setPars({1:'0.5 -1', 2:'0.5,,0.05,0.05,4.0,4.0',3 : '1.0', 4:'1,,0.05,0.05,4.0,4.0',5:'1.0' })
AllModels.show() # for checking

In [ ]:
# Fitting
Fit.statMethod = 'cstat' 
Fit.nIterations = 100  # fit iterations for each attempt
for delt in [1e-2, 1e-3, 1e-4, 1e-5]:
    Fit.delta = delt
    Fit.perform()

In [ ]:
for comp_name in mod1.componentNames: 
        comp = getattr(mod1, comp_name)
        for par_name in comp.parameterNames:
            param = getattr(comp, par_name)
            print(f'{par_name} : {param.values[0]}')

In [ ]:
diskbb_Tin = mod1.diskbb.Tin.values[0]
print(diskbb_Tin)
diskbb_norm = mod1.diskbb.norm.values[0]
print(diskbb_norm)
bbody_kT = mod1.bbodyrad.kT.values[0]
print(bbody_kT)
bbody_norm = mod1.bbodyrad.norm.values[0]
print(bbody_norm)

AllModels.clear()
AllModels.systematic = 0.00 # add systematic error
mod1 = Model('tbabs * cflux* (diskbb + bbodyrad)')
mod1.setPars({1:'0.5 -1', 2:1.0, 3:10.0, 4:'-10', 5:f'{diskbb_Tin},,0.05,0.05,4.0,4.0', 6 : f'{diskbb_norm} -1', 7:f'{bbody_kT},,0.05,0.05,4.0,4.0',8:f'{bbody_norm}' })
AllModels.show() # for checking

In [ ]:
Fit.statMethod = 'cstat' 
Fit.nIterations = 100  # fit iterations for each attempt
Fit.delta = 1e-5 # controls the convergence criterion for the fitting algorithm
Fit.perform()

In [ ]:
for comp_name in mod1.componentNames: 
        comp = getattr(mod1, comp_name)
        for par_name in comp.parameterNames:
            param = getattr(comp, par_name)
            print(f'{par_name} : {param.values[0]}')

In [ ]:
n_params = mod1.nParameters
Fit.error(f'maximum 3.0 1.0 1-{n_params}')

In [ ]:
par1 = mod1.cflux.lg10Flux 

# best-fit value
val1 = par1.values[0]
print(val1)

# bounds and ± errors
lo, hi, status = par1.error
print(lo,hi)
err_minus1 = val1 - lo
err_plus1  = hi - val1
print(err_minus1, err_plus1)

print(f"norm1 = {10**(val1)} (-{10**(val1) - 10**(val1 - err_minus1)}, +{10**(val1 + err_plus1) - 10**val1})")
print(f"norm1 = {10**(val1)} (-{10**(val1) - 10**(lo)}, +{10**(hi) - 10**val1})")

In [ ]:
print("chi2: ", Fit.testStatistic, " dof: ", Fit.dof, " redchi2: ", Fit.testStatistic/Fit.dof)

In [ ]:
plot_resid("", "", mod=mod1, save=False)

---

# XSPEC EXAMPLE CODE


```
 xspec
data ./spectra_swift_xrt/Obs_00033665136wtfinal_bin.pi
abund wilm
xsect vern
ignore **-0.6,10.0-**
notice 0.6-10.0
ignore bad
mo tbabs*(powerlaw+bbodyrad)
nh: 0.5 -1
Tin: 1.6,,0.05,0.05,4.0,4.0
norm: 1.0 
kT: 0.6,,0.05,0.05,4.0,4.0
norm: 1.0 

statistic cstat


newpar 0
fit 100 1e-2
fit 100 1e-3
fit 100 1e-4
fit 100 1e-5


editmod tbabs*cflux*(powerlaw+bbodyrad)
Emin: 1.0
Emax: 10.0
lg10Flux: -10
freeze 6
newpar 0
fit 100 1e-5


error 1.0 1-8
```

---